# Notebook 0 — Preprocesamiento de la serie de demanda eléctrica del SNI

**Proyecto:** Sistema de pronóstico de demanda eléctrica en Ecuador mediante modelos autorregresivos y redes neuronales recurrentes
**Autor:** Cristhian Andrés Sánchez Cevallos
**Tutor:** Ing. Freddy Tejada Escobar, M.Sc.

---

Este notebook implementa la **Fase 1** de la metodología:

1. Carga del CSV con los 60 valores mensuales (2020–2024)
2. Análisis exploratorio y visualización
3. Pruebas de estacionariedad (ADF y KPSS)
4. Descomposición estacional aditiva (s=12)
5. ACF y PACF de la serie diferenciada
6. Split cronológico 70/15/15
7. Escalado MinMax (ajustado solo con train)
8. Generación de ventanas para LSTM
9. Exportación de los artefactos para los notebooks 1, 2 y 3

**Importante:** ejecuta este notebook **primero**, antes de los notebooks 1, 2 y 3.


## 1. Instalación de dependencias (Colab)

In [ ]:
# Colab ya trae todo lo necesario. Si falta algo:
# !pip install statsmodels scikit-learn joblib pandas matplotlib seaborn --quiet
print("Entorno listo.")


## 2. Importaciones

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import MinMaxScaler
import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["font.family"] = "serif"


## 3. Carga del CSV

Sube a Colab el archivo `demanda_mensual_SNI_2020_2024.csv` que generamos en la fase de recopilación de datos.
Si estás en local, ajusta la ruta.


In [ ]:
# Opción A: subir desde Colab (descomenta si trabajas en Colab)
# from google.colab import files
# uploaded = files.upload()

RUTA_CSV = "demanda_mensual_SNI_2020_2024.csv"

df = pd.read_csv(RUTA_CSV, parse_dates=["fecha"]).sort_values("fecha").set_index("fecha")
serie = df["demanda_gwh"].astype(float).asfreq("MS")

print(f"Período: {serie.index.min().date()} a {serie.index.max().date()}")
print(f"Observaciones: {len(serie)}")
print(f"Faltantes: {int(serie.isna().sum())}")
print(f"Rango: {serie.min():.2f} - {serie.max():.2f} GWh")
print(f"Media: {serie.mean():.2f} GWh,  Desv.Std: {serie.std():.2f} GWh")
serie.head()


## 4. Visualización inicial de la serie completa

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
serie.plot(ax=ax, color="#1f4e79", marker="o", lw=1.5, markersize=4)
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-08-01"),
           alpha=0.18, color="tomato", label="COVID-19 (mar-ago 2020)")
ax.axvspan(pd.Timestamp("2024-09-01"), pd.Timestamp("2024-12-31"),
           alpha=0.18, color="orange", label="Estiaje 2024 (sep-dic)")
ax.set_title("Demanda eléctrica mensual del SNI (GWh), 2020-2024", fontsize=13)
ax.set_ylabel("GWh"); ax.set_xlabel("Fecha"); ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig01_serie_completa.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Estadísticas descriptivas anuales

In [ ]:
resumen = df.groupby(df.index.year)["demanda_gwh"].agg(["min", "max", "mean", "std", "sum"])
resumen.columns = ["Mínimo", "Máximo", "Media", "Desv.Std", "Total anual"]
resumen.index.name = "Año"
resumen = resumen.round(2)
print(resumen.to_string())


## 6. Pruebas de estacionariedad (ADF y KPSS)

- **ADF**: H0 = la serie tiene raíz unitaria (NO estacionaria). Si p < 0.05 → estacionaria.
- **KPSS**: H0 = la serie ES estacionaria. Si p < 0.05 → NO estacionaria.


In [ ]:
def test_adf(x, nombre):
    r = adfuller(x.dropna(), autolag="AIC")
    print(f"[ADF]  {nombre}: estadístico={r[0]:.4f}, p-valor={r[1]:.4f}",
          " => ESTACIONARIA" if r[1] < 0.05 else " => NO estacionaria")
    return r[1]

def test_kpss(x, nombre):
    r = kpss(x.dropna(), regression="c", nlags="auto")
    print(f"[KPSS] {nombre}: estadístico={r[0]:.4f}, p-valor={r[1]:.4f}",
          " => NO estacionaria" if r[1] < 0.05 else " => ESTACIONARIA")
    return r[1]

print("--- Serie original ---")
adf_orig = test_adf(serie, "original")
kpss_orig = test_kpss(serie, "original")

print("\n--- Diferenciación regular (d=1) ---")
serie_d1 = serie.diff().dropna()
adf_d1 = test_adf(serie_d1, "diff(1)")
kpss_d1 = test_kpss(serie_d1, "diff(1)")

print("\n--- Diferenciación estacional (D=1, s=12) ---")
serie_d12 = serie.diff(12).dropna()
if len(serie_d12) >= 12:
    test_adf(serie_d12, "diff(12)")
    test_kpss(serie_d12, "diff(12)")

print("\n--- Diferenciación regular + estacional ---")
serie_d1_d12 = serie.diff().diff(12).dropna()
if len(serie_d1_d12) >= 12:
    test_adf(serie_d1_d12, "diff(1)+diff(12)")
    test_kpss(serie_d1_d12, "diff(1)+diff(12)")


## 7. Descomposición estacional aditiva (s=12)

In [ ]:
descomp = seasonal_decompose(serie, model="additive", period=12, extrapolate_trend="freq")

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
descomp.observed.plot(ax=axes[0], color="#1f4e79"); axes[0].set_ylabel("Observado")
descomp.trend.plot(ax=axes[1], color="#2e7d32"); axes[1].set_ylabel("Tendencia")
descomp.seasonal.plot(ax=axes[2], color="#ef6c00"); axes[2].set_ylabel("Estacional")
descomp.resid.plot(ax=axes[3], color="#6a1b9a", marker="o", ms=3, lw=0.8); axes[3].set_ylabel("Residuo")
axes[3].axhline(0, color="gray", lw=0.5)
plt.suptitle("Descomposición aditiva (período s=12)", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig("fig02_descomposicion.png", dpi=300, bbox_inches="tight")
plt.show()


## 8. ACF y PACF (sobre la serie diferenciada)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(serie_d1, ax=axes[0], lags=min(24, len(serie_d1)//2 - 1),
         title="ACF de la serie diferenciada", color="#1f4e79")
plot_pacf(serie_d1, ax=axes[1], lags=min(24, len(serie_d1)//2 - 1),
          title="PACF de la serie diferenciada", color="#1f4e79", method="ywm")
plt.tight_layout()
plt.savefig("fig03_acf_pacf.png", dpi=300, bbox_inches="tight")
plt.show()


## 9. Split cronológico 70/15/15

Sin aleatorización (respetando el orden temporal):
- **Train:** ene-2020 a jun-2023 (42 obs)
- **Val:** jul-2023 a mar-2024 (9 obs)
- **Test:** abr-2024 a dic-2024 (9 obs)  ← incluye estiaje severo


In [ ]:
train = serie.iloc[:42]
val   = serie.iloc[42:51]
test  = serie.iloc[51:]

print(f"Train: {len(train):3d} obs   [{train.index.min().date()} -> {train.index.max().date()}]")
print(f"Val:   {len(val):3d} obs   [{val.index.min().date()} -> {val.index.max().date()}]")
print(f"Test:  {len(test):3d} obs   [{test.index.min().date()} -> {test.index.max().date()}]")
print(f"\nProporciones: {len(train)/len(serie):.1%} / {len(val)/len(serie):.1%} / {len(test)/len(serie):.1%}")

# Visualización del split
fig, ax = plt.subplots(figsize=(13, 4.5))
train.plot(ax=ax, label=f"Train ({len(train)} obs)", color="#1f4e79", lw=1.6, marker="o", ms=3)
val.plot(ax=ax,   label=f"Val ({len(val)} obs)",   color="#2e7d32", lw=1.6, marker="s", ms=3)
test.plot(ax=ax,  label=f"Test ({len(test)} obs)", color="#c62828", lw=1.6, marker="^", ms=3)
ax.set_title("División cronológica del conjunto de datos")
ax.set_ylabel("GWh"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig04_split.png", dpi=300, bbox_inches="tight")
plt.show()


## 10. Escalado MinMax (ajustado solo con train)

Importante: el escalador se ajusta **únicamente** con los datos de entrenamiento, para evitar la fuga de información del conjunto de prueba hacia el modelo.


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_s = scaler.fit_transform(train.values.reshape(-1, 1)).flatten()
val_s   = scaler.transform(val.values.reshape(-1, 1)).flatten()
test_s  = scaler.transform(test.values.reshape(-1, 1)).flatten()

print(f"Train escalado: min={train_s.min():.4f}, max={train_s.max():.4f}")
print(f"Val escalado:   min={val_s.min():.4f}, max={val_s.max():.4f}")
print(f"Test escalado:  min={test_s.min():.4f}, max={test_s.max():.4f}")
print("\n(Val y test pueden salir fuera de [0,1] si tienen valores extremos respecto al train, es esperado)")


## 11. Ventanas deslizantes para LSTM

Ventana de entrada = 12 meses (un año de historia), horizonte = 1 mes.
Para no perder muestras al pasar entre bloques, se prepone contexto del bloque anterior.


In [ ]:
VENTANA = 12
HORIZONTE = 1

def crear_ventanas(arr, ventana, horizonte):
    X, y = [], []
    for i in range(len(arr) - ventana - horizonte + 1):
        X.append(arr[i:i+ventana])
        y.append(arr[i+ventana:i+ventana+horizonte])
    return np.array(X), np.array(y)

val_ext  = np.concatenate([train_s[-VENTANA:], val_s])
test_ext = np.concatenate([val_s[-VENTANA:], test_s])

X_train, y_train = crear_ventanas(train_s,  VENTANA, HORIZONTE)
X_val,   y_val   = crear_ventanas(val_ext,  VENTANA, HORIZONTE)
X_test,  y_test  = crear_ventanas(test_ext, VENTANA, HORIZONTE)

X_train = X_train.reshape(-1, VENTANA, 1)
X_val   = X_val.reshape(-1, VENTANA, 1)
X_test  = X_test.reshape(-1, VENTANA, 1)

print(f"Ventanas LSTM (ventana={VENTANA} meses, horizonte={HORIZONTE}):")
print(f"  X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape}   y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}   y_test:  {y_test.shape}")


## 12. Exportación de artefactos

In [ ]:
# Series crudas (para ARIMA, SARIMA y Prophet)
pd.DataFrame({"fecha": train.index, "demanda_gwh": train.values}).to_csv("train.csv", index=False)
pd.DataFrame({"fecha": val.index,   "demanda_gwh": val.values}).to_csv("val.csv", index=False)
pd.DataFrame({"fecha": test.index,  "demanda_gwh": test.values}).to_csv("test.csv", index=False)

# Ventanas LSTM
np.savez_compressed("lstm_data.npz",
                    X_train=X_train, y_train=y_train,
                    X_val=X_val,     y_val=y_val,
                    X_test=X_test,   y_test=y_test,
                    ventana=VENTANA, horizonte=HORIZONTE)

# Scaler para invertir el escalado al evaluar
joblib.dump(scaler, "minmax_scaler.pkl")

print("Artefactos exportados:")
print("  train.csv, val.csv, test.csv   (para Notebooks 1 y 2)")
print("  lstm_data.npz                  (para Notebook 3)")
print("  minmax_scaler.pkl              (para Notebook 3 - invertir escalado)")
print("\nFiguras guardadas en disco: fig01..fig04")
print("\nNotebook 0 completado. Continúa con Notebook 1 (ARIMA/SARIMA).")
